# Curating mutations across all 151 genomes (Adding CDS and AA change effects) 

### Import Statements

In [1]:
import numpy as np
import pandas as pd
#import vcf
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import scipy.stats

%matplotlib inline

In [2]:
#from scipy import stats

In [3]:
import bioframe as bf

In [4]:
import io 
import pathlib

In [5]:
from Bio import SeqIO


#### Pandas Viewing Settings

In [6]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

### Import `gcutils` custom functions

In [7]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
    
from gcutils.general import parse_PAF_VarTSV, label_DF_ByOvrLapGenes

from gcutils.general import infer_SNP_CDS_Consequences, addCodonInfo_To_Var

# Import/parse processed H37rv genome annotations

In [8]:
RepoRef_Dir = "../../References"
ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"
H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"

H37Rv_GenomeAnnotations_Genes_And_IntergenicRegions_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.And.IntergenicRegions.tsv"    
H37Rv_GenomeAnnotations_Genes_And_IntergenicRegions_BED = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.And.IntergenicRegions.bed"


## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t")

RvID_To_Symbol_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['H37rv_GeneID', 'Symbol']].values)
Symbol_To_RvID_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'H37rv_GeneID']].values)

Symbol_To_FuncCat_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'Functional_Category']].values)
RvID_To_FuncCat_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['H37rv_GeneID', 'Functional_Category']].values)


# Parse H37Rv Reference sequences (Genome, Genes, Proteins)

## Parse H37Rv genome sequence (DNA)

In [9]:
H37rv_Ref_GBK_PATH = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.gbk"
H37Rv_FA = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.fasta"

H37Rv_Seq = SeqIO.read(H37Rv_FA, "fasta").seq
len(H37Rv_Seq)

4411532

## Parse H37Rv Protein (AA) and gene (DNA) sequences

In [10]:
O2_RefDir = "/n/data1/hms/dbmi/farhat/mm774/References"

MycoBrowser_RefFiles_Dir = f"{O2_RefDir}/190619_Mycobrowser_H37rv_ReferenceFiles"

H37Rv_Proteins_MycoBro_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"
H37Rv_Proteins_MycoBro_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.esxM_Added.fasta"
H37Rv_Proteins_NCBI_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"

H37RV_Genes_FA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_genes_v3.fasta"

H37Rv_FAA_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_proteins.faa"
H37Rv_FAA_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_proteins.esxM_Added.faa"
H37Rv_GBK_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_genomic.gbk"


In [11]:
!ls -1 $MycoBrowser_RefFiles_Dir

Mycobacterium_tuberculosis_H37Rv_genes_v3.fasta
Mycobacterium_tuberculosis_H37Rv_genome_v3.fasta
Mycobacterium_tuberculosis_H37Rv_genome_v3.fasta.fai
Mycobacterium_tuberculosis_H37Rv_gff_v3.gff
Mycobacterium_tuberculosis_H37Rv_gff_v3.REP13E12_Regions.gff
Mycobacterium_tuberculosis_H37Rv_proteins_v3.fasta
Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.esxM_Added.fasta
Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta
Mycobacterium_tuberculosis_H37Rv_txt_v3_PEPPE_subfamilies.txt
Mycobacterium_tuberculosis_H37Rv_txt_v3.txt.tsv


### Parse MycoBrowser Protein Seq Ref

In [12]:
dictOf_H37Rv_MycoBrow_ProtSeq = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_Proteins_MycoBro_FAA, "fasta"))):
    ShortID = record.name
    
    dictOf_H37Rv_MycoBrow_ProtSeq[ShortID] = record.seq


4091it [00:00, 54506.95it/s]


### Parse MycoBrowser Gene Seq Ref

In [13]:
dictOf_H37Rv_MycoBrow_Gene_Seq = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37RV_Genes_FA, "fasta"))):

    ShortID = record.name.split("|")[0]
    dictOf_H37Rv_MycoBrow_Gene_Seq[ShortID] = record.seq


4187it [00:00, 83403.63it/s]


In [14]:
list(dictOf_H37Rv_MycoBrow_Gene_Seq.keys())[:2]

['Rv3728', 'Rv3729']

### Parse NCBI Protein Seq Ref

In [15]:
dictOf_H37Rv_ProtSeq = {}
dictOf_H37Rv_ProtRecord = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_FAA_PATH, "fasta"))):
    Rec_Description = record.description
    dict_Attr = {}
    for i in Rec_Description.split(" "):
        ###Just looking for line with " " character (as key = value)
        if "=" in i:
            key = i.strip().split("=")[0].strip('"').strip('[')
            value = i.strip().split("=")[1].strip('"').strip(']')
            ###Put them in a dictionnary
            dict_Attr[key]=value
    
    ShortID = dict_Attr["locus_tag"]
    dictOf_H37Rv_ProtSeq[ShortID] = record.seq
    dictOf_H37Rv_ProtRecord[ShortID] = record


3907it [00:00, 87937.93it/s]


In [16]:
#dictOf_H37Rv_ProtSeq["Rv1196"]

In [17]:
#dictOf_H37Rv_ProtSeq["Rv1196"] == dictOf_H37Rv_MycoBrow_ProtSeq["Rv1196"]

In [18]:
# radA (Rv3585)

In [19]:
dictOf_H37Rv_ProtSeq["Rv3585"][434]

'G'

In [20]:
dictOf_H37Rv_MycoBrow_ProtSeq["Rv3585"][0]

'V'

In [21]:
# ligB (Rv3062)

In [22]:
dictOf_H37Rv_ProtSeq["Rv3062"][434]

'W'

In [23]:
dictOf_H37Rv_MycoBrow_ProtSeq["Rv3062"][434]

'W'

# Parse in H37Rv Homology-Map Results (k19w19)

### Define all HmMap file paths

In [24]:
#Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9"
Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V10"

H37_Rv_MM2_HomologyMapping_Dir = f"{Main_Project_Dir}/250901.H37Rv.HomologyMapping.k19w19.ProcessedData.V2"

# Define paths to output TSVS

### Homologous regions (MERGED)
RvHmMap_Merged_ParaRegions_TSV  = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.ParalogousRegions.k19w19.tsv"
RvHmMap_Merged_LocalRepeats_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.LocalRepeats.k19w19.tsv"

### Homology map (pairwise alignments)
RvHmMap_Aln_All_TSV           = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.All.tsv"
RvHmMap_Aln_PRs_NoOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.tsv"
RvHmMap_Aln_LRs_WiOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.WiOverlap.tsv"

RvHmMap_Aln_PRs_NoOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.Clustered.tsv"
RvHmMap_Aln_LRs_WiOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.OnlyOverlap.Clustered.tsv"

### Variants from the homology map alignments
RvHmMap_Var_TSV      = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.tsv"
RvHmMap_Var_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.snps.tsv"



### Variants from the homology map alignments (For each pairwise alignment between PRs)
RvHmMap_Var_PerAln_TSV      = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.ParalogousRegions.PerAln.tsv"
RvHmMap_Var_PerAln_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.ParalogousRegions.PerAln.snps.tsv"



### Parse in HmRegions (`Paralogous_Regions` and `Local_Repeats`)

In [25]:
HmMapRegs_ParaRegs_k19w19_DF = pd.read_csv(RvHmMap_Merged_ParaRegions_TSV,
                                           sep="\t")

HmMapRegs_ParaRegs_k19w19_DF.shape

(200, 15)

In [26]:
HmMapRegs_LocalRepeats_k19w19_DF = pd.read_csv(RvHmMap_Merged_LocalRepeats_TSV, 
                                               sep="\t")
HmMapRegs_LocalRepeats_k19w19_DF.shape

(50, 13)

In [27]:
HmMapRegs_All_LRsPRs_K19w19_DF = pd.concat([HmMapRegs_ParaRegs_k19w19_DF,
                                            HmMapRegs_LocalRepeats_k19w19_DF])

HmMapRegs_All_LRsPRs_K19w19_DF.shape

(250, 15)

### Parse in homology-map DFs (pairwise alignments between all homologous regions)

In [28]:
HmMap_Aln_k19w19_DF = pd.read_csv(RvHmMap_Aln_All_TSV,
                           sep="\t")
HmMap_Aln_k19w19_DF.shape

(776, 25)

In [29]:
HmMap_Aln_k19w19_NoOverlap_DF = pd.read_csv(RvHmMap_Aln_PRs_NoOverlap_TSV,
                                     sep="\t")
HmMap_Aln_k19w19_NoOverlap_DF.shape

(640, 38)

In [30]:
HmMap_Aln_k19w19_LocalRepeat_DF = pd.read_csv(RvHmMap_Aln_LRs_WiOverlap_TSV,
                                              sep="\t")
HmMap_Aln_k19w19_LocalRepeat_DF.shape

(136, 35)

### Parse in HomologyMap Alignment Variants DFs

In [31]:
### paralog variants - Improved per alignment `cs` tag approach 
Mtb_HM_Var_PR_DF = pd.read_csv(RvHmMap_Var_PerAln_TSV, sep="\t")
print(Mtb_HM_Var_PR_DF.shape)

Mtb_HM_Var_PR_SNPs_DF = pd.read_csv(RvHmMap_Var_PerAln_SNPs_TSV, sep="\t")
print(Mtb_HM_Var_PR_SNPs_DF.shape)

(60402, 20)
(50203, 20)


In [32]:
Mtb_HM_Var_PR_SNPs_DF.head(4)

,Target_Name,Target_Start,Target_End,Strand,Ref,Alt,SNP,Type,Query_Name,Query_Start,Query_End,HmMap_Aln_ID,Aln_Query_Start,Aln_Query_End,Aln_Target_Start,Aln_Target_End,Aln_Strand,QueryOverlap_Genes,TargetOverlap_Genes,QueryParalog_RegionID
0,NC_000962.3,3082657,3082658,+,T,C,True,SNP,NC_000962.3,80412,80413,NC_000962.3:80184-80523;NC_000962.3:3082465-30...,80184,80523,3082465,3082769,+,Rv0071,Rv2774c,Rv0071-NC_000962.3:80184-80523
1,NC_000962.3,3082660,3082661,+,C,A,True,SNP,NC_000962.3,80415,80416,NC_000962.3:80184-80523;NC_000962.3:3082465-30...,80184,80523,3082465,3082769,+,Rv0071,Rv2774c,Rv0071-NC_000962.3:80184-80523
2,NC_000962.3,3082667,3082668,+,G,A,True,SNP,NC_000962.3,80422,80423,NC_000962.3:80184-80523;NC_000962.3:3082465-30...,80184,80523,3082465,3082769,+,Rv0071,Rv2774c,Rv0071-NC_000962.3:80184-80523
3,NC_000962.3,3082669,3082670,+,T,C,True,SNP,NC_000962.3,80424,80425,NC_000962.3:80184-80523;NC_000962.3:3082465-30...,80184,80523,3082465,3082769,+,Rv0071,Rv2774c,Rv0071-NC_000962.3:80184-80523


In [33]:
TarCol = ['Target_Name', 'Target_Start', 'Target_End', 'Ref', 'Alt', 'SNP'] 

Mtb_HM_Var_PR_SNPs_Trim_DF = Mtb_HM_Var_PR_SNPs_DF[TarCol]

print(Mtb_HM_Var_PR_SNPs_Trim_DF.shape)

Mtb_HM_Var_PR_SNPs_Trim_TrimUnq_DF = Mtb_HM_Var_PR_SNPs_Trim_DF.drop_duplicates()

print(Mtb_HM_Var_PR_SNPs_Trim_TrimUnq_DF.shape)


(50203, 6)
(41454, 6)


In [34]:
Mtb_HM_Var_PR_SNPs_Trim_TrimUnq_DF.head(3)

,Target_Name,Target_Start,Target_End,Ref,Alt,SNP
0,NC_000962.3,3082657,3082658,T,C,True
1,NC_000962.3,3082660,3082661,C,A,True
2,NC_000962.3,3082667,3082668,G,A,True


# Parse `Mtb151CI` Isolate Metadata

In [35]:
Repo_DataDir = "../../Data"
InputAsmPath_Dir = f"{Repo_DataDir}/231121.InputAsmTSVs.MtbSetV3.151CI"

MtbSetV3_151CI_InputAsmPATHs_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAndSRAsm.FAPATHs.V1.tsv"
MtbSetV3_151CI_AsmSumm_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAsm.AsmSummary.V2.tsv"


### Reading in "WGA151CI_AsmSummary_DF"

In [36]:
WGA151CI_AsmSummary_DF = pd.read_csv(MtbSetV3_151CI_AsmSumm_TSV, sep = "\t")

SampleIDs_151CI_SOI = list( WGA151CI_AsmSummary_DF["SampleID"].values )
WGA151CI_SampleIDs = SampleIDs_151CI_SOI
WGA151CI_AsmSummary_DF.shape


(151, 7)

#### Create SampleID Mapping Dicts

In [37]:
WGA151CI_ID_To_PrimLineage_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'PrimaryLineage']].values)
WGA151CI_ID_To_SubLineage_Dict = dict( WGA151CI_AsmSummary_DF[["SampleID", "Lineage"]].values)
WGA151CI_ID_To_Dataset_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'Dataset_Tag']].values)  
ID_To_PrimLineage_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'PrimaryLineage']].values)
ID_To_SubLineage_Dict = dict( WGA151CI_AsmSummary_DF[["SampleID", "Lineage"]].values)
ID_To_Dataset_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'Dataset_Tag']].values)

#### Parse Assembly to FASTA DF

In [38]:
WGA151CI_Asm_Path_DF = pd.read_csv(MtbSetV3_151CI_InputAsmPATHs_TSV, sep = "\t")
ID_To_CompleteAsm_Dict = dict(WGA151CI_Asm_Path_DF[['SampleID', 'Genome_ASM_PATH']].values)  
WGA151CI_Asm_Path_DF.shape

(151, 4)

In [39]:
WGA151CI_Asm_Path_DF.head(2)

,SampleID,Dataset_Tag,Genome_ASM_PATH,ShortRead_Genome_ASM_PATH
0,N0072,ChinerOms_2019,/n/data1/hms/dbmi/farhat/mm774/Projects/231121...,/n/data1/hms/dbmi/farhat/mm774/Projects/231121...
1,N0153,ChinerOms_2019,/n/data1/hms/dbmi/farhat/mm774/Projects/231121...,/n/data1/hms/dbmi/farhat/mm774/Projects/231121...


In [40]:
WGA151CI_AsmSummary_DF["Dataset_Tag"].value_counts()

Dataset_Tag
Hall2022                78
TB_Portals_24CI_R1      21
Peker2021               17
Farhat_Peru_2019        13
ChinerOms_2019          12
TRUST_PB_Set1            8
Lee2020_Elife            1
Ngabonziza_Lin8_2020     1
Name: count, dtype: int64

# Parse `Nuc-Div` results for `WGA-151CI` dataset

## Define NucDiv analysis file paths

In [43]:
AnalysisName = "250201.WGA151CI.V10"

Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V10"

V8_Target_Output_Dir = f"{Main_Project_Dir}/{AnalysisName}"

VCFT_NucDiv_Dir = f"{V8_Target_Output_Dir}/VCFtools_SNV_NucDiv_AcrossH37Rv"

Pickle_PATH_WGA_NucDiv_PI_Dict = VCFT_NucDiv_Dir + f"/{AnalysisName}.SNVs.NucDiv.DictOfResults.pickle"   

#NucDiv_NoFilt_1kb_TSV_PATH = f"{VCFT_NucDiv_Dir}/WGA.SNVs.NucDiv.1000bp.ALL.tsv"
NucDiv_NoFilt_1kb_TSV_PATH = f"{VCFT_NucDiv_Dir}/WGA.SNVs.NucDiv.1000bp.ALL.Anno.tsv"
NucDiv_NoFilt_1kb_V2_TSV_PATH = f"{VCFT_NucDiv_Dir}/WGA.SNVs.NucDiv.1000bp.ALL.Anno.V2.tsv"


### Parse pickle of NucDiv stats

In [44]:
# with open(Pickle_PATH_WGA_NucDiv_PI_Dict, "rb") as f: NucDiv_PI_Dict = pickle.load(f)
# NucDiv_PI_Dict.keys()

In [45]:
# xx_1kb_All_np = NucDiv_PI_Dict[("ALL", 1000)]["Fill_np"]["XX"]
# yy_1kb_All_np = NucDiv_PI_Dict[("ALL", 1000)]["Fill_np"]["YY"] * 1000


### Parse 1-kb NucDiv DF

In [46]:
# Read in TSV
WGA_NucDiv_PI_1kb_DF = pd.read_csv(NucDiv_NoFilt_1kb_V2_TSV_PATH, sep = "\t")

NucDiv_1kb_DF = WGA_NucDiv_PI_1kb_DF

# Calculate mean, median, standard deviation, and Median Absolute Deviation
median_1kb_PI = NucDiv_1kb_DF['NucDiv'].median()
mean_1kb_PI = NucDiv_1kb_DF['NucDiv'].mean()
std_1kb_PI = NucDiv_1kb_DF['NucDiv'].std()
MAD_1kb_PI = np.median(np.abs(NucDiv_1kb_DF['NucDiv'] - median_1kb_PI))


### Define NucDiv Hospots (N = 37)

In [47]:
NucDiv_HSR_1kb_DF = NucDiv_1kb_DF.query("IsNucDivHotspot == True")
NucDiv_HSR_1kb_DF.shape

(37, 20)

# Begin variant annotation and processing

## Create dict of all variants detected (Var TSV) for each assembly

In [48]:
### Define directories to PMP-SM (PacBio assembly and analysis pipeline)

Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects"
Mtb_WGA_SMK_Outputs_Dir = Project_Dir + "/Mtb-WGA-SMK-Output"

WGA151CI_SMK_OutputDir = Mtb_WGA_SMK_Outputs_Dir + "/231121_MtbSetV3_151CI"

target_SMK_OutputDir = WGA151CI_SMK_OutputDir


In [49]:
listOfSample_Tags = WGA151CI_SampleIDs

dictOf_Var_PerAsm = {}

listOfAll_VarDFs = []

for SampleID in tqdm(listOfSample_Tags):
    
    dictOf_Var_PerAsm[SampleID] = {}
    
    sample_Asm_OutputDir = target_SMK_OutputDir + "/AsmAnalysis/" + SampleID

    i_VarCall_MainDir = f"{sample_Asm_OutputDir}/VariantCallingVersusH37Rv"

    Var_AsmToRv_Dir = f"{i_VarCall_MainDir}/MM2_AsmToH37rv"

    i_Asm_Var_TSV = f"{Var_AsmToRv_Dir}/{SampleID}.mm2.AsmToH37Rv.var.tsv"

    i_Var_DF = parse_PAF_VarTSV(i_Asm_Var_TSV)

    i_Var_DF["SampleID"] = SampleID
    i_Var_DF["Sublineage"] = ID_To_SubLineage_Dict[SampleID]
    
    i_Var_SNP_DF = i_Var_DF.query("SNP == True")

    dictOf_Var_PerAsm[SampleID]["AsmVar_DF"] = i_Var_DF
    dictOf_Var_PerAsm[SampleID]["AsmVar_SNP_DF"] = i_Var_SNP_DF

    listOfAll_VarDFs.append(i_Var_DF)

MtbRefSet_151CI_AllVar_DF = pd.concat(listOfAll_VarDFs)
MtbRefSet_151CI_AllVar_DF.shape

100%|██████████| 151/151 [00:05<00:00, 28.66it/s]


(260869, 16)

In [50]:
MtbRefSet_151CI_AllVar_DF.shape

(260869, 16)

In [51]:
MtbRefSet_151CI_AllVar_DF.head()

,VariantTag,Target_Name,Target_Start,Target_End,Cov,MapQ,Ref,Alt,Query_Name,Query_Start,Query_End,Strand,SNP,Type,SampleID,Sublineage
0,V,NC_000962.3,1976,1977,1,60,A,G,N0072_contig_2_pilon,1976,1977,+,True,SNP,N0072,"lineage1,lineage1.1,lineage1.1.2"
1,V,NC_000962.3,2531,2532,1,60,T,C,N0072_contig_2_pilon,2531,2532,+,True,SNP,N0072,"lineage1,lineage1.1,lineage1.1.2"
2,V,NC_000962.3,4012,4013,1,60,T,C,N0072_contig_2_pilon,4012,4013,+,True,SNP,N0072,"lineage1,lineage1.1,lineage1.1.2"
3,V,NC_000962.3,6111,6112,1,60,G,C,N0072_contig_2_pilon,6111,6112,+,True,SNP,N0072,"lineage1,lineage1.1,lineage1.1.2"
4,V,NC_000962.3,6123,6124,1,60,C,T,N0072_contig_2_pilon,6123,6124,+,True,SNP,N0072,"lineage1,lineage1.1,lineage1.1.2"


In [52]:
TarCol = ['SampleID', 'Target_Name', 'Target_Start', 'Target_End', 'Ref', 'Alt', 'SNP'] 
MRS_151CI_AllVar_V2_DF = MtbRefSet_151CI_AllVar_DF[TarCol]

NewCol = ['SampleID', 'Chrom', 'Start_0', 'End', 'Ref', 'Alt', 'SNP'] 
MRS_151CI_AllVar_V2_DF.columns = NewCol

MRS_151CI_AllVar_V2_DF.shape

(260869, 7)

In [53]:
MRS_151CI_AllVar_V2_DF.head()

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP
0,N0072,NC_000962.3,1976,1977,A,G,True
1,N0072,NC_000962.3,2531,2532,T,C,True
2,N0072,NC_000962.3,4012,4013,T,C,True
3,N0072,NC_000962.3,6111,6112,G,C,True
4,N0072,NC_000962.3,6123,6124,C,T,True


## Run CDS variant annotation for ALL variants detected (Across 151 genomes)

In [54]:
MRS_151CI_VarWiCodon_DF = addCodonInfo_To_Var(MRS_151CI_AllVar_V2_DF, H37Rv_GenomeAnno_Genes_DF)
MRS_151CI_VarWiCodon_DF.shape

260869it [09:05, 478.40it/s]


(260869, 13)

In [55]:
MRS_151CI_MutConsequences_DF = infer_SNP_CDS_Consequences(MRS_151CI_VarWiCodon_DF,
                                                         dictOf_H37Rv_MycoBrow_Gene_Seq,
                                                         Symbol_To_RvID_Dict)


  1%|          | 931/130490 [00:00<01:13, 1756.68it/s]/home/mm774/miniforge/envs/bfds_v1/lib/python3.10/site-packages/Bio/Seq.py:2880: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
  1%|          | 1370/130490 [00:00<01:06, 1946.05it/s]

Error translating and inferring AA changes for ('01_R1430', 'mas') - mas - Rv2940c - 1 mutations to be inserted


  3%|▎         | 4005/130490 [00:02<00:55, 2275.05it/s]

Error translating and inferring AA changes for ('02_R1896', 'mas') - mas - Rv2940c - 1 mutations to be inserted


  4%|▎         | 4751/130490 [00:02<00:55, 2271.92it/s]

...

 99%|█████████▉| 128894/130490 [00:55<00:00, 2090.92it/s]

Error translating and inferring AA changes for ('mada_2-42', 'mas') - mas - Rv2940c - 1 mutations to be inserted


100%|█████████▉| 130161/130490 [00:56<00:00, 2296.19it/s]

Error translating and inferring AA changes for ('mada_2-46', 'mas') - mas - Rv2940c - 1 mutations to be inserted


100%|██████████| 130490/130490 [00:56<00:00, 2308.06it/s]


In [56]:
MRS_151CI_MutConsequences_DF.head()

,SampleID,Symbol,Codon,Ref_AA,Mut_AA
0,01_R1134,PE3,14,T,A
1,01_R1134,PE35,99,E,*
2,01_R1134,PE_PGRS1,346,R,G
3,01_R1134,PE_PGRS10,225,R,G
4,01_R1134,PE_PGRS10,227,R,G


In [57]:
MRS_151CI_MutConsequences_DF.shape

(118963, 5)

In [58]:
MRS_151CI_VarAnno_DF = pd.merge(MRS_151CI_VarWiCodon_DF,
                                MRS_151CI_MutConsequences_DF,
                                on = ["SampleID", "Symbol", "Codon"], how = "left")

print(MRS_151CI_VarAnno_DF.shape)

# SNP categorization based on Mut_AA being NaN (synonymous) or not (non-synonymous)
MRS_151CI_VarAnno_DF.loc[(MRS_151CI_VarAnno_DF['SNP'] == True) & (MRS_151CI_VarAnno_DF['Mut_AA'].isna()), 'Variant_Type'] = 'Synonymous_SNP'
MRS_151CI_VarAnno_DF.loc[(MRS_151CI_VarAnno_DF['SNP'] == True) & (MRS_151CI_VarAnno_DF['Mut_AA'].notna()), 'Variant_Type'] = 'NonSynonymous_SNP'     
MRS_151CI_VarAnno_DF.loc[(MRS_151CI_VarAnno_DF['SNP'] == False), 'Variant_Type'] = 'INDEL'  

print(MRS_151CI_VarAnno_DF.shape)

MRS_151CI_VarAnno_DF['Alt_WithStopCodon'] = MRS_151CI_VarAnno_DF['Mut_AA'].str.contains(r'\*').fillna("_")

MRS_151CI_VarAnno_DF['INDEL_Type'] = np.where(
    (MRS_151CI_VarAnno_DF['SNP'] == False) & (MRS_151CI_VarAnno_DF['Alt'].str.len() > MRS_151CI_VarAnno_DF['Ref'].str.len()), 'INS',
    np.where((MRS_151CI_VarAnno_DF['SNP'] == False) & (MRS_151CI_VarAnno_DF['Ref'].str.len() > MRS_151CI_VarAnno_DF['Alt'].str.len()), 'DEL', None)
)

# Create the 'frameshift' column with the specific conditions for INS and DEL
MRS_151CI_VarAnno_DF['frameshift'] = np.where(
    ((MRS_151CI_VarAnno_DF['INDEL_Type'] == 'INS') & (MRS_151CI_VarAnno_DF['Alt'].str.len() % 3 != 0)) |
    ((MRS_151CI_VarAnno_DF['INDEL_Type'] == 'DEL') & (MRS_151CI_VarAnno_DF['Ref'].str.len() % 3 != 0)),
    True,
    False
)

print(MRS_151CI_VarAnno_DF.shape)

(260869, 15)
(260869, 16)
(260869, 19)


#### Peak at SNPs anno by CDS Effect DF

In [59]:
MRS_151CI_VarAnno_DF.head(2)

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Variant_Type,Alt_WithStopCodon,INDEL_Type,frameshift
0,N0072,NC_000962.3,1976,1977,A,G,True,0,None,NaN,NaN,NaN,NaN,NaN,NaN,Synonymous_SNP,_,None,False
1,N0072,NC_000962.3,2531,2532,T,C,True,1,dnaN,+,480.0,161.0,1.0,NaN,NaN,Synonymous_SNP,_,None,False


In [60]:
MRS_151CI_VarAnno_DF.query("SampleID == 'mada_1-41' & Symbol == 'dnaN' ")

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Variant_Type,Alt_WithStopCodon,INDEL_Type,frameshift
216018,mada_1-41,NC_000962.3,2221,2222,C,T,True,1,dnaN,+,170.0,57.0,3.0,NaN,NaN,Synonymous_SNP,_,None,False
216019,mada_1-41,NC_000962.3,3185,3186,A,G,True,1,dnaN,+,1134.0,379.0,1.0,N,D,NonSynonymous_SNP,False,None,False


In [61]:
MRS_151CI_VarAnno_DF.query("Symbol == 'dnaN' ").head(4)

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Variant_Type,Alt_WithStopCodon,INDEL_Type,frameshift
1,N0072,NC_000962.3,2531,2532,T,C,True,1,dnaN,+,480.0,161.0,1.0,NaN,NaN,Synonymous_SNP,_,None,False
2751,N0153,NC_000962.3,2531,2532,T,C,True,1,dnaN,+,480.0,161.0,1.0,NaN,NaN,Synonymous_SNP,_,None,False
65397,N1272,NC_000962.3,2531,2532,T,C,True,1,dnaN,+,480.0,161.0,1.0,NaN,NaN,Synonymous_SNP,_,None,False
65398,N1272,NC_000962.3,3191,3192,A,G,True,1,dnaN,+,1140.0,381.0,1.0,N,D,NonSynonymous_SNP,False,None,False


In [62]:
MRS_151CI_VarAnno_DF.query("SampleID == 'mada_1-41' & Symbol == 'dnaN' ")

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Variant_Type,Alt_WithStopCodon,INDEL_Type,frameshift
216018,mada_1-41,NC_000962.3,2221,2222,C,T,True,1,dnaN,+,170.0,57.0,3.0,NaN,NaN,Synonymous_SNP,_,None,False
216019,mada_1-41,NC_000962.3,3185,3186,A,G,True,1,dnaN,+,1134.0,379.0,1.0,N,D,NonSynonymous_SNP,False,None,False


In [63]:
MRS_151CI_MutConsequences_DF.query("SampleID == 'mada_1-41' & Symbol == 'dnaN' ")

,SampleID,Symbol,Codon,Ref_AA,Mut_AA
98997,mada_1-41,dnaN,379,N,D


In [64]:
MRS_151CI_MutConsequences_DF.query("Symbol == 'dnaN' ")

,SampleID,Symbol,Codon,Ref_AA,Mut_AA
24117,N1176,dnaN,381,N,D
28113,N1272,dnaN,381,N,D
98997,mada_1-41,dnaN,379,N,D
116183,mada_2-31,dnaN,379,N,D


## Divide annotated variants by their predicted effect (Round 2)

In [65]:
Var_SNPs_Syn_DF =  MRS_151CI_VarAnno_DF.query("Variant_Type == 'Synonymous_SNP' ")
Var_SNPs_Syn_DF["Variant_Group"] = 'Synonymous_SNP'

Var_SNPs_NS_DF =  MRS_151CI_VarAnno_DF.query("Variant_Type == 'NonSynonymous_SNP' & Alt_WithStopCodon == False ")
Var_SNPs_NS_DF["Variant_Group"] = 'Missensse_SNP'

Var_SNPs_NS_WiSTOP_DF =  MRS_151CI_VarAnno_DF.query("Variant_Type == 'NonSynonymous_SNP' & Alt_WithStopCodon == True ")
Var_SNPs_NS_WiSTOP_DF["Variant_Group"] = 'PrematureStop_SNP'

Var_INDELs_Inframe_DF =  MRS_151CI_VarAnno_DF.query("Variant_Type == 'INDEL' & frameshift == False ")
Var_INDELs_Inframe_DF["Variant_Group"] = 'Inframe_INDEL'

Var_INDELs_Frameshift_DF =  MRS_151CI_VarAnno_DF.query("Variant_Type == 'INDEL' & frameshift == True ")
Var_INDELs_Frameshift_DF["Variant_Group"] = 'Frameshift_INDEL'


MRS_151CI_VarAnno_V2_DF = pd.concat([Var_SNPs_Syn_DF,
                                     Var_SNPs_NS_DF,
                                     Var_SNPs_NS_WiSTOP_DF, 
                                     Var_INDELs_Inframe_DF,
                                     Var_INDELs_Frameshift_DF ], axis = 0).sort_values(["SampleID", "Start_0", "End"]).reset_index(drop=True)

MRS_151CI_VarAnno_V2_DF.shape

/tmp/ipykernel_247725/1276963313.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Var_SNPs_Syn_DF["Variant_Group"] = 'Synonymous_SNP'
/tmp/ipykernel_247725/1276963313.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Var_SNPs_NS_DF["Variant_Group"] = 'Missensse_SNP'
/tmp/ipykernel_247725/1276963313.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas

(260869, 20)

#### Preview new DF

In [66]:
MRS_151CI_VarAnno_V2_DF.head()

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Variant_Type,Alt_WithStopCodon,INDEL_Type,frameshift,Variant_Group
0,01_R1134,NC_000962.3,1976,1977,A,G,True,0,None,NaN,NaN,NaN,NaN,NaN,NaN,Synonymous_SNP,_,None,False,Synonymous_SNP
1,01_R1134,NC_000962.3,4012,4013,T,C,True,1,recF,+,733.0,245.0,2.0,I,T,NonSynonymous_SNP,False,None,False,Missensse_SNP
2,01_R1134,NC_000962.3,7361,7362,G,C,True,1,gyrA,+,60.0,21.0,1.0,E,Q,NonSynonymous_SNP,False,None,False,Missensse_SNP
3,01_R1134,NC_000962.3,7584,7585,G,C,True,1,gyrA,+,283.0,95.0,2.0,S,T,NonSynonymous_SNP,False,None,False,Missensse_SNP
4,01_R1134,NC_000962.3,9303,9304,G,A,True,1,gyrA,+,2002.0,668.0,2.0,G,D,NonSynonymous_SNP,False,None,False,Missensse_SNP


In [67]:
MRS_151CI_VarAnno_V2_DF["Variant_Group"].value_counts()

Variant_Group
Missensse_SNP        121836
Synonymous_SNP        96849
Inframe_INDEL         28878
Frameshift_INDEL      11561
PrematureStop_SNP      1745
Name: count, dtype: int64

In [68]:
MRS_151CI_VarAnno_V2_DF["Variant_Group"].value_counts().sum()

260869

In [69]:
MRS_151CI_VarAnno_V2_DF.shape

(260869, 20)

# Measure inter-SNP distances for all variants in the same genome

### Split All Variants Detected into SNPs & INDELs

In [70]:
WGA151_AllVar_SNPs_V2_DF = MRS_151CI_AllVar_V2_DF.query("SNP == True")
WGA151_AllVar_INDELs_V2_DF = MRS_151CI_AllVar_V2_DF.query("SNP == False")

In [71]:
WGA151_AllVar_SNPs_V2_DF.head(2)

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP
0,N0072,NC_000962.3,1976,1977,A,G,True
1,N0072,NC_000962.3,2531,2532,T,C,True


In [72]:
Z = bf.closest(WGA151_AllVar_SNPs_V2_DF, df2=None,
               cols1 = ("SampleID", "Start_0", "End"),
               cols2 = ("SampleID", "Start_0", "End"),
               suffixes=('', '_2'))

# List columns to drop
columns_to_drop = [col for col in Z.columns if col.endswith('_2')]

# Drop these columns and rename "distance" column
WGA151_AllVar_SNPs_InterSNPDist_DF = Z.drop(columns=columns_to_drop).rename(columns={'distance': 'DistToNearestSNP'})


HmAlnPAF_CoordCols = ("Query_Name", "Query_Start", "Query_End")


WGA151_InterSNPDist_DF = bf.count_overlaps(WGA151_AllVar_SNPs_InterSNPDist_DF,
                                  HmMap_Aln_k19w19_NoOverlap_DF,
                                  cols1 = ("Chrom", "Start_0", "End"),
                                  cols2 = HmAlnPAF_CoordCols ).rename(columns={'count': 'N_HmMapAln_PR_Ovrlap'})

WGA151_InterSNPDist_DF["OvrlapWi_HmMapAln_PR"] = np.where(WGA151_InterSNPDist_DF['N_HmMapAln_PR_Ovrlap'] > 0, 1, 0)

WGA151_InterSNPDist_DF["RegionType"] = np.where(WGA151_InterSNPDist_DF['N_HmMapAln_PR_Ovrlap'] > 0, "PR", "Unq").astype(str)


WGA151_InterSNPDist_DF = bf.count_overlaps(WGA151_InterSNPDist_DF,
                                  NucDiv_HSR_1kb_DF,
                                  cols1 = ("Chrom", "Start_0", "End"),
                                  cols2 = ("Chrom", "Start", "End") ).rename(columns={'count': 'NucDivHotspot_Ovrlap'})

WGA151_InterSNPDist_DF["NucDivHotspot_Ovrlap"] = WGA151_InterSNPDist_DF['NucDivHotspot_Ovrlap'].astype(bool)

WGA151_InterSNPDist_DF["LogDist"] = np.log10(WGA151_InterSNPDist_DF["DistToNearestSNP"])


WGA151_InterSNPDist_DF.shape

/home/mm774/miniforge/envs/bfds_v1/lib/python3.10/site-packages/pandas/core/arrays/masked.py:672: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs2, **kwargs)


(220430, 13)

# Outputs TSVs of annotated and processed variant info 

In [73]:
Mtb151_AllVar_Dir = "../../Data/241030.Mtb151CI.AllVariants.Anno.V1"

Mtb151_AllVar_TSVGZ              = f"{Mtb151_AllVar_Dir}/Mtb.151CI.AllVariants.V1.tsv.gz"
Mtb151_AllVar_WiCDSAnno_TSVGZ    = f"{Mtb151_AllVar_Dir}/Mtb.151CI.AllVariants.WiCDSAnno.V1.tsv.gz"
Mtb151_SNPs_WiInterSNPDist_TSVGZ =  f"{Mtb151_AllVar_Dir}/Mtb.151CI.AllSNPs.WiInterSNPDists.V1.tsv.gz"

!mkdir $Mtb151_AllVar_Dir

In [74]:
!ls -1 $Mtb151_AllVar_Dir

## a) Output TSV - All variants (No CDS Anno)

In [75]:

MRS_151CI_AllVar_V2_DF.to_csv(Mtb151_AllVar_TSVGZ,
                     sep = "\t", compression='gzip',
                     index=False)

In [76]:
!ls -1 ../../Data/241030.Mtb151CI.AllVariants.Anno.V1/

Mtb.151CI.AllVariants.V1.tsv.gz


## b) Output TSV - All variants annotated by `CDS Effect`

In [77]:

MRS_151CI_VarAnno_V2_DF.to_csv(Mtb151_AllVar_WiCDSAnno_TSVGZ,
                                 sep = "\t", compression='gzip',
                                 index=False)

In [78]:
!pwd

/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/mtb-geneconv-manuscript/Analysis/3_Mtb_NucDivAnalysis_V5


In [79]:
!ls -1 ../../Data/241030.Mtb151CI.AllVariants.Anno.V1/

Mtb.151CI.AllVariants.V1.tsv.gz
Mtb.151CI.AllVariants.WiCDSAnno.V1.tsv.gz


In [80]:
!ls -lah ../../Data/241030.Mtb151CI.AllVariants.Anno.V1/

total 12M
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Mar 10 18:29 .
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Mar 10 18:29 ..
-rw-r--r-- 1 mm774 hpc_farhat 4.4M Mar 10 18:29 Mtb.151CI.AllVariants.V1.tsv.gz
-rw-r--r-- 1 mm774 hpc_farhat 7.2M Mar 10 18:29 Mtb.151CI.AllVariants.WiCDSAnno.V1.tsv.gz


## c) Output TSV - SNPs w/ Inter-SNP Distance TSV

In [81]:

WGA151_InterSNPDist_DF.to_csv(Mtb151_SNPs_WiInterSNPDist_TSVGZ,
                     sep = "\t",
                     index=False)


In [82]:
!ls -1 ../../Data/241030.Mtb151CI.AllVariants.Anno.V1/


Mtb.151CI.AllSNPs.WiInterSNPDists.V1.tsv.gz
Mtb.151CI.AllVariants.V1.tsv.gz
Mtb.151CI.AllVariants.WiCDSAnno.V1.tsv.gz


In [83]:
!ls -lah ../../Data/241030.Mtb151CI.AllVariants.Anno.V1/


total 15M
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Mar 10 18:29 .
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Mar 10 18:29 ..
-rw-r--r-- 1 mm774 hpc_farhat 3.4M Mar 10 18:29 Mtb.151CI.AllSNPs.WiInterSNPDists.V1.tsv.gz
-rw-r--r-- 1 mm774 hpc_farhat 4.4M Mar 10 18:29 Mtb.151CI.AllVariants.V1.tsv.gz
-rw-r--r-- 1 mm774 hpc_farhat 7.2M Mar 10 18:29 Mtb.151CI.AllVariants.WiCDSAnno.V1.tsv.gz


# Extra exploration and validation of results

In [84]:
# T_DF = addCodonInfo_To_Var(MRS_151CI_AllVar_V2_DF.query("SampleID == 'mada_1-41' & Start_0 == 2221 "), H37Rv_GenomeAnno_Genes_DF)
# T_DF.shape

In [85]:
#T_DF

In [86]:
MRS_151CI_VarAnno_DF.query("SampleID == 'mada_1-41' & Start_0 == 2221 ")

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Variant_Type,Alt_WithStopCodon,INDEL_Type,frameshift
216018,mada_1-41,NC_000962.3,2221,2222,C,T,True,1,dnaN,+,170.0,57.0,3.0,NaN,NaN,Synonymous_SNP,_,None,False


In [87]:
MRS_151CI_VarAnno_DF.query(" Start_0 == 3049060 ")

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Variant_Type,Alt_WithStopCodon,INDEL_Type,frameshift
75666,N1177,NC_000962.3,3049060,3049061,G,A,True,2,recX,-,25.0,9.0,2.0,S,L,NonSynonymous_SNP,False,None,False


In [88]:
# TCG gets mutated to TTG at 9th codon of recX

In [89]:
dictOf_H37Rv_ProtSeq["Rv0002"][57]

'G'

In [90]:
dictOf_H37Rv_ProtSeq["Rv0002"][57]

'G'

In [91]:
dictOf_H37Rv_ProtSeq["Rv0002"][:10]

Seq('MDAATTRVGL')

In [92]:
dictOf_H37Rv_ProtSeq["Rv0002"][56]

'S'

In [93]:
171.0 / 3

57.0

In [94]:
2221 - 2051

170

In [95]:
MRS_151CI_VarAnno_DF.query("SNP == True").shape

(220430, 19)

In [96]:
MRS_151CI_VarAnno_DF.query("SNP == False").shape

(40439, 19)

In [97]:
MRS_151CI_VarAnno_DF.query("Symbol == 'ppsA'").shape

(2298, 19)

In [98]:
MRS_151CI_VarAnno_DF.query("Symbol == 'ppsA'").head(25)

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Variant_Type,Alt_WithStopCodon,INDEL_Type,frameshift
1895,N0072,NC_000962.3,3247088,3247089,G,A,True,1,ppsA,+,1644.0,549.0,1.0,G,S,NonSynonymous_SNP,False,None,False
1896,N0072,NC_000962.3,3247315,3247316,C,G,True,1,ppsA,+,1871.0,624.0,3.0,D,E,NonSynonymous_SNP,False,None,False
1897,N0072,NC_000962.3,3247850,3247851,G,A,True,1,ppsA,+,2406.0,803.0,1.0,A,T,NonSynonymous_SNP,False,None,False
1898,N0072,NC_000962.3,3247852,3247853,C,T,True,1,ppsA,+,2408.0,803.0,3.0,A,T,NonSynonymous_SNP,False,None,False
1899,N0072,NC_000962.3,3247855,3247856,G,C,True,1,ppsA,+,2411.0,804.0,3.0,NaN,NaN,Synonymous_SNP,_,None,False
1900,N0072,NC_000962.3,3247864,3247865,G,T,True,1,ppsA,+,2420.0,807.0,3.0,NaN,NaN,Synonymous_SNP,_,None,False
1901,N0072,NC_000962.3,3247865,3247866,C,A,True,1,ppsA,+,2421.0,808.0,1.0,Q,R,NonSynonymous_SNP,False,None,False
1902,N0072,NC_000962.3,3247866,3247867,A,G,True,1,ppsA,+,2422.0,808.0,2.0,Q,R,NonSynonymous_SNP,False,None,False
1903,N0072,NC_000962.3,3247867,3247868,A,G,True,1,ppsA,+,2423.0,808.0,3.0,Q,R,NonSynonymous_SNP,False,None,False
1904,N0072,NC_000962.3,3247868,3247869,A,G,True,1,ppsA,+,2424.0,809.0,1.0,N,D,NonSynonymous_SNP,False,None,False


In [99]:
MRS_151CI_VarWiCodon_DF.head(3)

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos
0,N0072,NC_000962.3,1976,1977,A,G,True,0,None,NaN,NaN,NaN,NaN
1,N0072,NC_000962.3,2531,2532,T,C,True,1,dnaN,+,480.0,161.0,1.0
2,N0072,NC_000962.3,4012,4013,T,C,True,1,recF,+,733.0,245.0,2.0


In [100]:
MRS_151CI_VarWiCodon_DF.query("SampleID == 'mada_1-41' & Symbol == 'dnaN' ")

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos
1,mada_1-41,NC_000962.3,2221,2222,C,T,True,1,dnaN,+,170.0,57.0,3.0
2,mada_1-41,NC_000962.3,3185,3186,A,G,True,1,dnaN,+,1134.0,379.0,1.0


In [101]:
MRS_151CI_VarWiCodon_DF.query("Symbol == 'dnaN' ").head(3)

,SampleID,Chrom,Start_0,End,Ref,Alt,SNP,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos
1,N0072,NC_000962.3,2531,2532,T,C,True,1,dnaN,+,480.0,161.0,1.0
2,N0153,NC_000962.3,2531,2532,T,C,True,1,dnaN,+,480.0,161.0,1.0
2,N1272,NC_000962.3,2531,2532,T,C,True,1,dnaN,+,480.0,161.0,1.0
